### Requirements:
- [gdown](https://github.com/wkentaro/gdown) (For download to datalake to clean)
- [dotenv](pypi.org/project/python-dotenv/) (For password and suchs)
- [Pocketbase Python SDK](https://github.com/vaphes/pocketbase)

Create a new `.env` file with the same structure as `example.env`, change email, password to Pocketbase login info

## 1. Download file from datalake

In [2]:
import gdown

In [ ]:
id = "1LfAv_C1biyjBNCjq8i5H_aAiKZX6Wy39"
output = "./datalake/"
gdown.download_folder(id=id, output=output)

## 2. Pocketbase

In [21]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# Get variables:
# os.getenv("VARIABLE_NAME", "")

True

In [22]:
from pocketbase import PocketBase

Login first:

In [23]:
client = PocketBase(os.getenv("POCKETBASE_URL", ""))

admin_data = client.collection("_superusers").auth_with_password(os.getenv("POCKETBASE_EMAIL", ""), os.getenv("POCKETBASE_PASSWORD", ""));

admin_data.is_valid

True

### 2.1. Upload data to database

In [2]:
import json

# Open and read the JSON file
with open('hardcover_1-10.json', encoding='utf-8') as file:
    data = json.load(file)

In [9]:
data_old = data
data = data_old[0]

In [17]:
data["dto_combined"]["isbn_10"]

'0590353403'

In [19]:
result = {
    "id": data["dto_combined"]["isbn_13"],
    "ISBN_10": data["dto_combined"]["isbn_10"],
    "title": data["dto_combined"]["title"],
    "description": data["dto_combined"]["description"],
    "image_url": data["cached_image"]["url"],
    "author": data["cached_contributors"][0]["author"]["name"],
    "language": data["dto_combined"]["language_id"],
    "publish_date": data["dto_combined"]["release_date"],
    "num_pages": data["dto_combined"]["page_count"],
    "crawl_source": "hardcover",
    "series": [series for series in data["dto_combined"]["series"]],
    "genre": [tag["tag"] for tag in data["cached_tags"]["Genre"]]
}

In [20]:
result

{'id': '9780590353403',
 'ISBN_10': '0590353403',
 'title': "Harry Potter and the Sorcerer's Stone",
 'description': "Harry Potter has never been the star of a Quidditch team, scoring points while riding a broom far above the ground. He knows no spells, has never helped to hatch a dragon, and has never worn a cloak of invisibility.\n\nAll he knows is a miserable life with the Dursleys, his horrible aunt and uncle, and their abominable son, Dudley - a great big swollen spoiled bully. Harry's room is a tiny closet at the foot of the stairs, and he hasn't had a birthday party in eleven years.\n\nBut all that is about to change when a mysterious letter arrives by owl messenger: a letter with an invitation to an incredible place that Harry - and anyone who reads about him - will find unforgettable.\n\nFor it's there that he finds not only friends, aerial sports, and magic in everything from classes to meals, but a great destiny that's been waiting for him... if Harry can survive the encount

In [29]:
for genre in result['genre']:
    try:
        record = client.collection('genre').create({ 'id': genre.lower().replace(' ', '-')[:15],'name': genre });
        print(record.__dict__)
    except Exception as e:
        print(f"Error processing '{genre}': {e}")


Error processing 'Fantasy': Response error. Status code:400
Error processing 'Fiction': Response error. Status code:400
Error processing 'Young Adult': Response error. Status code:400
Error processing 'Adventure': Response error. Status code:400
Error processing 'Classics': Response error. Status code:400
{'id': 'juvenile-fictio', 'created': datetime.datetime(2024, 12, 13, 12, 40, 57), 'updated': '', 'expand': {}, 'collection_id': 'pbc_1209437459', 'collection_name': 'genre', 'name': 'Juvenile Fiction'}
{'id': 'nonfiction', 'created': datetime.datetime(2024, 12, 13, 12, 40, 57), 'updated': '', 'expand': {}, 'collection_id': 'pbc_1209437459', 'collection_name': 'genre', 'name': 'Nonfiction'}
{'id': 'mystery', 'created': datetime.datetime(2024, 12, 13, 12, 40, 58), 'updated': '', 'expand': {}, 'collection_id': 'pbc_1209437459', 'collection_name': 'genre', 'name': 'Mystery'}
{'id': 'general', 'created': datetime.datetime(2024, 12, 13, 12, 40, 58), 'updated': '', 'expand': {}, 'collection_

In [33]:
result['genre'] = [name.lower().replace(' ', '-')[:15] for name in result['genre']]

In [34]:
result

{'id': '9780590353403',
 'ISBN_10': '0590353403',
 'title': "Harry Potter and the Sorcerer's Stone",
 'description': "Harry Potter has never been the star of a Quidditch team, scoring points while riding a broom far above the ground. He knows no spells, has never helped to hatch a dragon, and has never worn a cloak of invisibility.\n\nAll he knows is a miserable life with the Dursleys, his horrible aunt and uncle, and their abominable son, Dudley - a great big swollen spoiled bully. Harry's room is a tiny closet at the foot of the stairs, and he hasn't had a birthday party in eleven years.\n\nBut all that is about to change when a mysterious letter arrives by owl messenger: a letter with an invitation to an incredible place that Harry - and anyone who reads about him - will find unforgettable.\n\nFor it's there that he finds not only friends, aerial sports, and magic in everything from classes to meals, but a great destiny that's been waiting for him... if Harry can survive the encount

In [36]:
result

{'id': '9780590353403',
 'ISBN_10': '0590353403',
 'title': "Harry Potter and the Sorcerer's Stone",
 'description': "Harry Potter has never been the star of a Quidditch team, scoring points while riding a broom far above the ground. He knows no spells, has never helped to hatch a dragon, and has never worn a cloak of invisibility.\n\nAll he knows is a miserable life with the Dursleys, his horrible aunt and uncle, and their abominable son, Dudley - a great big swollen spoiled bully. Harry's room is a tiny closet at the foot of the stairs, and he hasn't had a birthday party in eleven years.\n\nBut all that is about to change when a mysterious letter arrives by owl messenger: a letter with an invitation to an incredible place that Harry - and anyone who reads about him - will find unforgettable.\n\nFor it's there that he finds not only friends, aerial sports, and magic in everything from classes to meals, but a great destiny that's been waiting for him... if Harry can survive the encount

In [37]:
record = client.collection('book').create(result);
record.__dict__

{'id': '9780590353403',
 'created': datetime.datetime(2024, 12, 13, 12, 45, 22),
 'updated': datetime.datetime(2024, 12, 13, 12, 45, 22),
 'expand': {},
 'isbn_10': '0590353403',
 'author': 'J. K. Rowling',
 'average_rating': 0,
 'collection_id': 'pbc_3974677633',
 'collection_name': 'book',
 'crawl_source': 'hardcover',
 'description': "Harry Potter has never been the star of a Quidditch team, scoring points while riding a broom far above the ground. He knows no spells, has never helped to hatch a dragon, and has never worn a cloak of invisibility.\n\nAll he knows is a miserable life with the Dursleys, his horrible aunt and uncle, and their abominable son, Dudley - a great big swollen spoiled bully. Harry's room is a tiny closet at the foot of the stairs, and he hasn't had a birthday party in eleven years.\n\nBut all that is about to change when a mysterious letter arrives by owl messenger: a letter with an invitation to an incredible place that Harry - and anyone who reads about him 

In [46]:
ISBN_13 = "9789295055025"

book_data = {
    "id": ISBN_13,
    "ISBN_10": "9295055020",
    "title": "A history",
    "description": "test",
    "image_url": "https://example.com",
    "author": "test",
    "language": "test",
    "publish_date": "2022-01-01 10:00:00.123Z",
    "num_pages": 123,
    "crawl_source": "goodreads",
    # "series": "RELATION_RECORD_ID",
    # "genre": [
    #     "RELATION_RECORD_ID"
    # ],
    "average_rating": 123,
    "review_count": 123
};

record = client.collection('book').create(book_data);
record.__dict__

# If you run this code again, it will produce error because you can't insert 2 objects with same id into table

{'id': '9789295055025',
 'created': datetime.datetime(2024, 12, 12, 20, 6, 13),
 'updated': datetime.datetime(2024, 12, 12, 20, 6, 13),
 'expand': {},
 'isbn_10': '9295055020',
 'author': 'test',
 'average_rating': 123,
 'collection_id': 'pbc_3974677633',
 'collection_name': 'book',
 'crawl_source': 'goodreads',
 'description': 'test',
 'genre': [],
 'image_url': 'https://example.com',
 'language': 'test',
 'num_pages': 123,
 'publish_date': '2022-01-01 10:00:00.123Z',
 'review_count': 123,
 'series': '',
 'title': 'A history'}

### 2.2. Get data from database

In [53]:
result = client.collection("book").get_full_list()

print(result)
result[0].__dict__

[<Record: 9789295055025>]


{'id': '9789295055025',
 'created': datetime.datetime(2024, 12, 12, 20, 6, 13),
 'updated': datetime.datetime(2024, 12, 12, 20, 6, 13),
 'expand': {},
 'isbn_10': '9295055020',
 'author': 'test',
 'average_rating': 123,
 'collection_id': 'pbc_3974677633',
 'collection_name': 'book',
 'crawl_source': 'goodreads',
 'description': 'test',
 'genre': [],
 'image_url': 'https://example.com',
 'language': 'test',
 'num_pages': 123,
 'publish_date': '2022-01-01 10:00:00.123Z',
 'review_count': 123,
 'series': '',
 'title': 'A history'}

In [ ]:

# Parameters:
# result = client.collection("book").get_full_list(query_params = {
#     "fields": "id,ISBN_10,title,autho"
# })